# 📈 Visualización de Datos — TFG Crypto

Este notebook genera gráficas exploratorias a partir de los CSVs semanales
creados por **`extraccion_datos.ipynb`**.

> ⚠️ **Ejecuta primero `extraccion_datos.ipynb`** para generar los archivos CSV.

**Gráficas incluidas:**
1. Precio de cierre semanal (BTC y ETH)
2. Volumen semanal (BTC y ETH)
3. Fear & Greed Index
4. RSI-14 (BTC y ETH)
5. MACD (BTC)
6. Bandas de Bollinger (BTC)
7. Volatilidad histórica 4w (BTC y ETH)

In [ ]:
# ============================================================
# CELDA 1 — Imports y carga de datos
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

# ── Ruta de los CSVs ────────────────────────────────────────
RUTA_CSV = Path(r"C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv")

# ── Estilo global de las gráficas ───────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
COLOR_BTC  = "#F7931A"   # naranja Bitcoin
COLOR_ETH  = "#627EEA"   # azul/morado Ethereum
COLOR_MIED = "#CC2200"   # rojo  (miedo extremo)
COLOR_COD  = "#00AA44"   # verde (codicia extrema)

# ── Cargar CSVs ─────────────────────────────────────────────
btc = pd.read_csv(RUTA_CSV / "btc-usd_semanal.csv",    index_col="date", parse_dates=True)
eth = pd.read_csv(RUTA_CSV / "eth-usd_semanal.csv",    index_col="date", parse_dates=True)
fg  = pd.read_csv(RUTA_CSV / "fear_greed_semanal.csv", index_col="date", parse_dates=True)

print("✅ Datos cargados correctamente")
print(f"   BTC: {len(btc)} semanas ({btc.index[0].date()} → {btc.index[-1].date()})")
print(f"   ETH: {len(eth)} semanas ({eth.index[0].date()} → {eth.index[-1].date()})")
print(f"   F&G: {len(fg)}  semanas ({fg.index[0].date()}  → {fg.index[-1].date()})")

---
## Gráfica 1 — Precio de Cierre Semanal

In [ ]:
# ============================================================
# CELDA 2 — Precio de cierre BTC y ETH (doble eje Y)
# ============================================================
# BTC (naranja) en el eje izquierdo, ETH (azul) en el derecho.
# Escala logarítmica para comparar mejor los ciclos de mercado.

fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.set_title("Precio de Cierre Semanal — BTC y ETH (escala log)",
              fontsize=14, fontweight="bold", pad=14)
ax1.semilogy(btc.index, btc["close"], color=COLOR_BTC, linewidth=1.8,
             label="BTC-USD (eje izq.)")
ax1.set_ylabel("BTC-USD (USD)", color=COLOR_BTC, fontsize=11)
ax1.tick_params(axis="y", labelcolor=COLOR_BTC)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

ax2 = ax1.twinx()
ax2.semilogy(eth.index, eth["close"], color=COLOR_ETH, linewidth=1.8,
             linestyle="--", label="ETH-USD (eje der.)")
ax2.set_ylabel("ETH-USD (USD)", color=COLOR_ETH, fontsize=11)
ax2.tick_params(axis="y", labelcolor=COLOR_ETH)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.savefig(RUTA_CSV.parent / "figuras" / "01_precio_cierre.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("✅ Gráfica 1 — Precio de cierre")

---
## Gráfica 2 — Volumen Semanal

In [ ]:
# ============================================================
# CELDA 3 — Volumen semanal BTC y ETH
# ============================================================
# Barras semitransparentes para BTC y ETH con doble eje Y.

fig, ax1 = plt.subplots(figsize=(14, 4))

ax1.set_title("Volumen Semanal — BTC y ETH",
              fontsize=14, fontweight="bold", pad=14)

width = 4  # días de ancho de cada barra
ax1.bar(btc.index, btc["volume"], width=width,
        color=COLOR_BTC, alpha=0.7, label="BTC (eje izq.)")
ax1.set_ylabel("Volumen BTC (USD)", color=COLOR_BTC, fontsize=10)
ax1.tick_params(axis="y", labelcolor=COLOR_BTC)
ax1.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x/1e9:.0f}B"))

ax2 = ax1.twinx()
ax2.bar(eth.index + pd.Timedelta(days=4), eth["volume"], width=width,
        color=COLOR_ETH, alpha=0.7, label="ETH (eje der.)")
ax2.set_ylabel("Volumen ETH (USD)", color=COLOR_ETH, fontsize=10)
ax2.tick_params(axis="y", labelcolor=COLOR_ETH)
ax2.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x/1e9:.0f}B"))

ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()
print("✅ Gráfica 2 — Volumen")

---
## Gráfica 3 — Fear & Greed Index

In [ ]:
# ============================================================
# CELDA 4 — Fear & Greed Index con zonas de color
# ============================================================
# Zonas:
#   0–25  → Miedo Extremo  (rojo)
#   25–45 → Miedo          (naranja)
#   45–55 → Neutral        (amarillo)
#   55–75 → Codicia        (verde claro)
#   75–100 → Codicia Extrema (verde)

fig, ax = plt.subplots(figsize=(14, 4))

ax.set_title("Fear & Greed Index — Semanal (media)",
             fontsize=14, fontweight="bold", pad=14)

# Rellenar zonas
ax.axhspan(0,  25,  color="#CC2200", alpha=0.12, label="Miedo extremo (0–25)")
ax.axhspan(25, 45,  color="#FF6600", alpha=0.12, label="Miedo (25–45)")
ax.axhspan(45, 55,  color="#FFCC00", alpha=0.14, label="Neutral (45–55)")
ax.axhspan(55, 75,  color="#88CC44", alpha=0.12, label="Codicia (55–75)")
ax.axhspan(75, 100, color="#00AA44", alpha=0.12, label="Codicia extrema (75–100)")

# Línea del índice
ax.plot(fg.index, fg["fear_greed"], color="#0055AA",
        linewidth=1.6, label="Fear & Greed (media semanal)")

# Líneas de referencia
ax.axhline(y=25, color="#CC2200", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axhline(y=75, color="#00AA44", linestyle="--", linewidth=0.8, alpha=0.6)

ax.set_ylim(0, 100)
ax.set_ylabel("Índice (0 = Miedo · 100 = Codicia)", fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper right", fontsize=8, ncol=2)

plt.tight_layout()
plt.show()
print("✅ Gráfica 3 — Fear & Greed")

---
## Gráfica 4 — RSI-14

In [ ]:
# ============================================================
# CELDA 5 — RSI-14 para BTC y ETH
# ============================================================
# Líneas de referencia:
#   70 → zona de sobrecompra
#   30 → zona de sobreventa

fig, ax = plt.subplots(figsize=(14, 4))

ax.set_title("RSI-14 Semanal — BTC y ETH",
             fontsize=14, fontweight="bold", pad=14)

# Zonas visuales
ax.axhspan(70, 100, color="#CC2200", alpha=0.08, label="Sobrecompra (>70)")
ax.axhspan(0,  30,  color="#00AA44", alpha=0.08, label="Sobreventa (<30)")

ax.plot(btc.index, btc["rsi_14"], color=COLOR_BTC,
        linewidth=1.6, label="RSI-14 BTC")
ax.plot(eth.index, eth["rsi_14"], color=COLOR_ETH,
        linewidth=1.6, linestyle="--", label="RSI-14 ETH")

# Líneas de referencia
ax.axhline(y=70, color="#CC2200", linestyle=":", linewidth=1.2)
ax.axhline(y=50, color="gray",    linestyle=":", linewidth=0.8)
ax.axhline(y=30, color="#00AA44", linestyle=":", linewidth=1.2)

ax.set_ylim(0, 100)
ax.set_ylabel("RSI (0–100)", fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()
print("✅ Gráfica 4 — RSI-14")

---
## Gráfica 5 — MACD (BTC)

In [ ]:
# ============================================================
# CELDA 6 — MACD de BTC (histograma + líneas MACD y señal)
# ============================================================
# Histograma verde cuando MACD > señal (momentum alcista)
# Histograma rojo  cuando MACD < señal (momentum bajista)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7),
                                gridspec_kw={"height_ratios": [2, 1]},
                                sharex=True)

# Panel superior — precio de cierre BTC
ax1.set_title("MACD (12, 26, 9) — BTC Semanal",
              fontsize=14, fontweight="bold", pad=14)
ax1.semilogy(btc.index, btc["close"], color=COLOR_BTC, linewidth=1.5)
ax1.semilogy(btc.index, btc["ema_20"], color="gray",
             linewidth=1.0, linestyle="--", label="EMA-20")
ax1.set_ylabel("Precio BTC (USD, log)", fontsize=10)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax1.legend(loc="upper left")

# Panel inferior — MACD
colores_hist = ["#00AA44" if v >= 0 else "#CC2200"
                for v in btc["macd_hist"].fillna(0)]
ax2.bar(btc.index, btc["macd_hist"], width=5,
        color=colores_hist, alpha=0.7, label="Histograma")
ax2.plot(btc.index, btc["macd"],       color="#0055AA",
         linewidth=1.5, label="MACD")
ax2.plot(btc.index, btc["macd_senal"], color="#FF6600",
         linewidth=1.5, label="Señal")
ax2.axhline(y=0, color="gray", linewidth=0.8)
ax2.set_ylabel("MACD", fontsize=10)
ax2.legend(loc="upper left")

ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()
print("✅ Gráfica 5 — MACD BTC")

---
## Gráfica 6 — Bandas de Bollinger (BTC)

In [ ]:
# ============================================================
# CELDA 7 — Bandas de Bollinger BTC (20 semanas, ±2σ)
# ============================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7),
                                gridspec_kw={"height_ratios": [3, 1]},
                                sharex=True)

# Panel superior — precio + bandas
ax1.set_title("Bandas de Bollinger (20 semanas, ±2σ) — BTC",
              fontsize=14, fontweight="bold", pad=14)
ax1.semilogy(btc.index, btc["close"],       color=COLOR_BTC, linewidth=1.5, label="Close BTC")
ax1.semilogy(btc.index, btc["bb_media"],    color="gray",    linewidth=1.0,
             linestyle="--", label="SMA-20 (media)")
ax1.semilogy(btc.index, btc["bb_superior"], color="#CC2200", linewidth=0.8,
             label="Banda superior (+2σ)")
ax1.semilogy(btc.index, btc["bb_inferior"], color="#00AA44", linewidth=0.8,
             label="Banda inferior (−2σ)")
ax1.fill_between(btc.index, btc["bb_superior"], btc["bb_inferior"],
                 alpha=0.07, color="gray")
ax1.set_ylabel("Precio BTC (USD, log)", fontsize=10)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax1.legend(loc="upper left", fontsize=9)

# Panel inferior — ancho de banda (%)
ax2.plot(btc.index, btc["bb_ancho"], color="#7B2FBE", linewidth=1.4)
ax2.set_ylabel("Amplitud (%)\n[(sup−inf)/media]", fontsize=9)
ax2.set_xlabel("")

ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()
print("✅ Gráfica 6 — Bollinger BTC")

---
## Gráfica 7 — Volatilidad Histórica Rolling (4 semanas)

In [ ]:
# ============================================================
# CELDA 8 — Volatilidad histórica BTC y ETH (4 semanas rolling)
# ============================================================
# Cuanto mayor el valor, más volátil fue el mercado esa semana.

fig, ax = plt.subplots(figsize=(14, 4))

ax.set_title("Volatilidad Histórica Rolling (4 semanas) — BTC y ETH",
             fontsize=14, fontweight="bold", pad=14)

ax.plot(btc.index, btc["volatilidad_4w"], color=COLOR_BTC,
        linewidth=1.6, label="Volatilidad BTC")
ax.plot(eth.index, eth["volatilidad_4w"], color=COLOR_ETH,
        linewidth=1.6, linestyle="--", label="Volatilidad ETH")

ax.set_ylabel("Volatilidad (% ret. log semanal)", fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper right")
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()
print("✅ Gráfica 7 — Volatilidad")
print("\n🎉 ¡Todas las gráficas generadas!")